In [1]:
import os
import warnings
import numpy as np
import optuna
import time
import sys

In [2]:
import hyperparameters
from utils import RandomTrial, read_data
from modelsnnpc import *

In [3]:
from optuna.storages import RetryFailedTrialCallback
from optuna.pruners import ThresholdPruner
from optuna.samplers import TPESampler

In [4]:
BASE_SEED = 42
np.random.seed(BASE_SEED)

In [5]:
def optimize_parameters(trial, dataset_name, train_dfs, test_dfs, hyperparameters, layers):
    betas = tuple(
        trial.suggest_float(f'beta{i+1}', 0.1, 1.0, log=True) for i in range(layers)
    ) if hyperparameters['beta'] is None else hyperparameters['beta']
    slope = trial.suggest_int('slope', 10, 50, step=1) if hyperparameters['slope'] is None else hyperparameters['slope']
    thresholds=tuple(
        trial.suggest_float(f'threshold{i+1}', 0.1, 1, log=True) for i in range(layers)
    ) if hyperparameters['threshold'] is None else hyperparameters['threshold']
    weight_minority_class = trial.suggest_float('weight', 0.95, 1, log=True) if hyperparameters['weight'] is None else hyperparameters['weight']
    class_weights = (1-weight_minority_class, weight_minority_class) 
    adam_betas = tuple( 
        trial.suggest_float(f'adam_beta{i+1}', 0.97, 0.99, step=0.001) for i in range(2)
    ) if hyperparameters['adam_beta'] is None else hyperparameters['adam_beta']
    learning_rate = trial.suggest_float('learning_rate', 1e-6, 1e-3, log=True) if hyperparameters['learning_rate'] is None else hyperparameters['learning_rate']
    train_df = train_dfs[dataset_name].iloc[:, :32]
    test_df = test_dfs[dataset_name].iloc[:, :32]
    x_train = train_df.drop(columns=["fraud_bool"])
    y_train = train_df["fraud_bool"]
    x_test = test_df.drop(columns=["fraud_bool"])
    y_test = test_df["fraud_bool"]
    num_classes = len(np.unique(y_train))
    num_features = len(x_train.columns)
    model = ModelSNNPC(
        num_features=num_features,
        num_classes=num_classes,
        population=hyperparameters['population'],
        class_weights=class_weights,
        betas=betas,
        slope=slope,
        thresholds=thresholds,
        batch_size=hyperparameters['batch'],
        num_epochs=hyperparameters['epoch'],
        num_steps=hyperparameters['step'],
        adam_betas=adam_betas,
        learning_rate=learning_rate,
        verbose=0
    )
    fit_time = time.time()
    model.fit(x_train, y_train)
    trial.set_user_attr("@time train", time.time()-fit_time)
    inference_time = time.time()
    predictions, targets = model.predict(x_test, y_test)
    trial.set_user_attr("@time inference", time.time()-inference_time)
    metrics = model.evaluate(targets, predictions)
    metrics_business = model.evaluate_business_constraint(targets, predictions)
    metrics.update(metrics_business)
    fairness_age = model.evaluate_fairness(x_test, targets, predictions, "customer_age", 50)
    metrics.update({k+"_age": v for k, v in fairness_age.items()})
    fairness_income = model.evaluate_fairness(x_test, targets, predictions, "income", 0.5)
    metrics.update({k+"_income": v for k, v in fairness_income.items()})
    fairness_employement = model.evaluate_fairness(x_test, targets, predictions, "employment_status", 3)
    metrics.update({k+"_employment": v for k, v in fairness_employement.items()})
    trial.set_user_attr("@global accuracy", metrics["accuracy"])
    trial.set_user_attr("@global precision", metrics["precision"])
    trial.set_user_attr("@global recall", metrics["recall"])
    trial.set_user_attr("@global fpr", metrics["fpr"])
    trial.set_user_attr("@global f1_score", metrics["f1_score"])
    trial.set_user_attr("@global auc", metrics["auc"])
    try:
        trial.set_user_attr("@5FPR fpr", metrics["fpr@5FPR"])
        trial.set_user_attr("@5FPR recall", metrics["recall@5FPR"])
        trial.set_user_attr("@5FPR accuracy", metrics["accuracy@5FPR"])
        trial.set_user_attr("@5FPR precision", metrics["precision@5FPR"])
        trial.set_user_attr("@5FPR threshold", metrics["threshold"])
        # age attributes
        trial.set_user_attr("@5FPR fpr_ratio_age", metrics["fpr_ratio_age"])
        trial.set_user_attr("@5FPR fnr_ratio_age", metrics["fnr_ratio_age"])
        trial.set_user_attr("@5FPR eod_age", metrics["eod_age"])
        trial.set_user_attr("@5FPR aod_age", metrics["aod_age"])
        # income attributes
        trial.set_user_attr("@5FPR fpr_ratio_income", metrics["fpr_ratio_income"])
        trial.set_user_attr("@5FPR fnr_ratio_income", metrics["fnr_ratio_income"])
        trial.set_user_attr("@5FPR eod_income", metrics["eod_income"])
        trial.set_user_attr("@5FPR aod_income", metrics["aod_income"])
        # employment attributes
        trial.set_user_attr("@5FPR fpr_ratio_employment", metrics["fpr_ratio_employment"])
        trial.set_user_attr("@5FPR fnr_ratio_employment", metrics["fnr_ratio_employment"])
        trial.set_user_attr("@5FPR eod_employment", metrics["eod_employment"])
        trial.set_user_attr("@5FPR aod_employment", metrics["aod_employment"])
    except Exception:
        pass
    objectives = [metrics[y] for (_,y) in OBJECTIVE]
    return objectives

In [6]:
def main(datasets_list, study_name, trials_optuna, sampler, objective, hyperparameters, layers=3):
    base_path = "/kaggle/input/bank-account-fraud-dataset/"
    _, datasets, train_dfs, test_dfs = read_data(base_path, datasets_list, seed=BASE_SEED)
    for dataset_name in datasets.keys(): 
        # storage = optuna.storages.RDBStorage(
        #     url="sqlite:///ciarp.db",
        #     heartbeat_interval=60,
        #     grace_period=120,
        #     failed_trial_callback=RetryFailedTrialCallback(max_retry=3),
        # ) 
        study = optuna.create_study(
            directions=[x for (x,_) in objective],
            study_name=f"{study_name}",
            sampler=sampler,
            pruner=ThresholdPruner(lower=0.01, upper=0.99)
        )
        study.optimize(lambda trial, dataset_name=dataset_name: optimize_parameters(trial, dataset_name, train_dfs, test_dfs, hyperparameters, layers), n_trials=trials_optuna)
        study.trials_dataframe().to_csv(f"/kaggle/working/study_{study_name}_results.csv", index=False)
        try:
            print(study.best_params)
            print(study.best_value)
            print(study.best_trial)
        except Exception:
            pass

In [7]:
ps_stepper = [{'population': 2,
              'step': 50},
             {'population': 20,
              'step': 50},
             {'population': 200,
              'step': 50},
             {'population': 2,
              'step': 20},
             {'population': 20,
              'step': 20},
             {'population': 200,
              'step': 20}]

for ps in ps_stepper[4:6]:
    HYPERPARAMETERS = {
    "population": ps['population'],
    "batch": 3072, #1024,
    "epoch": 5,
    "step":  ps['step'],
    "beta": None,
    "slope": None,
    "threshold": None,
    "weight": None,
    "adam_beta": None,
    "learning_rate": None,
    }
    LAYERS = 4
    DATASETS = ["Variant I"]
    STUDY_NAME = f"CIARP2024-P{HYPERPARAMETERS['population']}-S{HYPERPARAMETERS['step']}"
    # TRIALS_OPTUNA = 1050
    TRIALS_OPTUNA = 10
    SAMPLER = TPESampler()
    OBJECTIVE = [("minimize","aod_age"), ("maximize","recall")]
    main(DATASETS, STUDY_NAME, TRIALS_OPTUNA, SAMPLER, OBJECTIVE, HYPERPARAMETERS, LAYERS)

[I 2025-04-05 19:20:23,843] A new study created in memory with name: CIARP2024-P20-S20
[I 2025-04-05 19:30:42,978] Trial 0 finished with values: [0.0, 0.3439888811674774] and parameters: {'beta1': 0.3059315772032638, 'beta2': 0.32892617635579663, 'beta3': 0.45974249245094434, 'beta4': 0.20340150901300383, 'slope': 32, 'threshold1': 0.23709523472527835, 'threshold2': 0.13882507811156217, 'threshold3': 0.3550219320177619, 'threshold4': 0.8007600244729697, 'weight': 0.9756549094004425, 'adam_beta1': 0.97, 'adam_beta2': 0.99, 'learning_rate': 4.271839306085592e-05}.
[I 2025-04-05 19:41:01,846] Trial 1 finished with values: [0.0, 0.31758165392633775] and parameters: {'beta1': 0.6155023216787111, 'beta2': 0.37366627831306387, 'beta3': 0.3871465780044055, 'beta4': 0.23612016470171232, 'slope': 16, 'threshold1': 0.38089193979759367, 'threshold2': 0.11935127299071929, 'threshold3': 0.6991224083030745, 'threshold4': 0.10138270852054847, 'weight': 0.9620400950036412, 'adam_beta1': 0.984, 'adam_be